In [12]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support, precision_score, recall_score, f1_score, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.impute import KNNImputer
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

In [13]:
DATA_PATH_T = "True.csv"
DATA_PATH_F = "Fake.csv"

df_t = pd.read_csv(DATA_PATH_T)
print(df_t.shape)
display(df_t.head(5))

df_f = pd.read_csv(DATA_PATH_F)
print(df_f.shape)
display(df_f.head(5))

(21417, 4)


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


(23481, 4)


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [14]:
df_t = df_t.copy()
df_f = df_f.copy()

df_t["label"] = "True"
df_f["label"] = "Fake"

df_all = pd.concat([df_t, df_f], ignore_index=True)

In [15]:
import re
from collections import Counter
import pandas as pd

def normalize_and_tokenize(text: str):
    """
    Rules:
    - lowercase
    - keep numbers
    - treat hyphens/spaces between alphanumerics as equivalent (merge them)
      so: high-speed, high speed, highspeed -> highspeed
    - raw tokens are alphanumeric runs after normalization
    """
    if pd.isna(text):
        return []
    s = str(text).lower()

    # 1) Merge "word[- ]word" patterns repeatedly:
    #    - "high-speed" -> "highspeed"
    #    - "high speed" -> "highspeed"
    #    - "covid-19" -> "covid19"
    #    - "covid 19" -> "covid19"
    # Apply repeatedly to handle multi-part phrases like "state-of-the-art" -> "stateoftheart"
    pattern = re.compile(r"([a-z0-9]+)[-\s]+(?=[a-z0-9]+)")
    prev = None
    while prev != s:
        prev = s
        s = pattern.sub(r"\1", s)

    # 2) Tokenize: keep only letters+digits as tokens (punctuation becomes separators)
    tokens = re.findall(r"[a-z0-9]+", s)
    return tokens


def top_words(df: pd.DataFrame, text_col: str, top_n: int = 20, min_token_len: int = 1):
    counter = Counter()
    for doc in df[text_col].fillna(""):
        tokens = normalize_and_tokenize(doc)
        if min_token_len > 1:
            tokens = [t for t in tokens if len(t) >= min_token_len]
        counter.update(tokens)

    return pd.DataFrame(counter.most_common(top_n), columns=["word", "count"])

## At our own discretion, we will list the top 10 words of each excluding tokens like "the", "is", "he says", etc.

In [6]:
TOP_N = 50          # set to 10, 20, 100, etc.
MIN_TOKEN_LEN = 2   # optional: helps remove single-letter noise like "u", "s"

true_title_top = top_words(df_t, "title", top_n=TOP_N, min_token_len=MIN_TOKEN_LEN)
fake_title_top = top_words(df_f, "title", top_n=TOP_N, min_token_len=MIN_TOKEN_LEN)

true_text_top  = top_words(df_t, "text",  top_n=TOP_N, min_token_len=MIN_TOKEN_LEN)
fake_text_top  = top_words(df_f, "text",  top_n=TOP_N, min_token_len=MIN_TOKEN_LEN)

display(true_title_top)
display(fake_title_top)
display(true_text_top)
display(fake_text_top)

,word,count
0,trump,454
1,factbox,409
2,whitehouse,275
3,exclusive,208
4,sources,167
5,russia,119
6,official,118
7,china,115
8,source,102
9,trumpontwitter,98


,word,count
0,video,7779
1,watch,1222
2,breaking,702
3,trump,493
4,tweets,472
5,wow,371
6,obama,268
7,here,243
8,it,240
9,details,213


,word,count
0,reuters,21324
1,washington,6953
2,hesaid,5937
3,trump,3377
4,theu,3203
5,however,2241
6,it,1884
7,shesaid,1574
8,we,1373
9,newyork,1082


,word,count
0,com,7973
1,twitter,6711
2,however,3852
3,via,3464
4,co,3331
5,gettyimages,3210
6,2017,3024
7,trump,2926
8,https,2793
9,pic,2601


In [20]:
import pandas as pd

# --- 1) Parse dates + label ---
df_t = df_t.copy()
df_f = df_f.copy()

df_t["label"] = "True"
df_f["label"] = "Fake"

df_t["date_parsed"] = pd.to_datetime(df_t["date"], errors="coerce", infer_datetime_format=True)
df_f["date_parsed"] = pd.to_datetime(df_f["date"], errors="coerce", infer_datetime_format=True)

df_all = pd.concat([df_t, df_f], ignore_index=True)

# --- 2) Filter to requested date range ---
start_date = pd.Timestamp("2015-03-30")
end_date   = pd.Timestamp("2018-02-18")

df_all = df_all.dropna(subset=["date_parsed"]).copy()
df_all = df_all[(df_all["date_parsed"] >= start_date) & (df_all["date_parsed"] <= end_date)]

# --- 3) Count articles by month and label ---
df_all["month"] = df_all["date_parsed"].dt.to_period("M").astype(str)  # e.g., "2017-12"

monthly_counts = (
    df_all.groupby(["month", "label"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure both columns exist even if a dataset is missing in a month
for col in ["Fake", "True"]:
    if col not in monthly_counts.columns:
        monthly_counts[col] = 0

# Order columns + sort by month
monthly_counts = monthly_counts[["month", "Fake", "True"]].sort_values("month")

# Optional: include total
monthly_counts["Total"] = monthly_counts["Fake"] + monthly_counts["True"]

monthly_counts

/var/folders/ph/604hyll972n5jvrgj31cvqcc0000gn/T/ipykernel_92414/3414998388.py:10: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_t["date_parsed"] = pd.to_datetime(df_t["date"], errors="coerce", infer_datetime_format=True)
/var/folders/ph/604hyll972n5jvrgj31cvqcc0000gn/T/ipykernel_92414/3414998388.py:11: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_f["date_parsed"] = pd.to_datetime(df_f["date"], errors="coerce", infer_datetime_format=True)


label,month,Fake,True,Total
0,2015-05,338,0,338
1,2016-01,695,246,941
2,2016-02,687,432,1119
3,2016-03,679,490,1169
4,2016-04,610,383,993
5,2016-05,1012,394,1406
6,2016-06,477,419,896
7,2016-07,465,338,803
8,2016-08,438,265,703
9,2016-09,486,351,837
